# Accuracy Analysis

This notebook analyzes the accuracy of Gemini 3.5 Flash and Qwen 2.5 7B (SLM) overall and across different trip planning complexities (number of cities).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import os

# Set style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
colors_3 = ["#1f77b4", "#ff7f0e", "#2ca02c"] # Blue, Orange, Green

## 1. Overall Accuracy Comparison

First, we load the overall accuracy metrics from the evaluation results and compare them.

In [ ]:
with open('../data/gemini3-flash/gemini3_evaluation_results.json', 'r') as f:
    gemini_results = json.load(f)

with open('../data/qwen2.5-7b/qwen2.5_evaluation_results.json', 'r') as f:
    qwen_results = json.load(f)

# Global Accuracy from Summaries
acc_gemini = gemini_results['summary']['Total_Accuracy_%']
acc_slm_pass1 = qwen_results['summary']['Total_Pass@1_Accuracy_%']
acc_slm_pass5 = qwen_results['summary']['Total_Pass@5_Accuracy_%']

df_standard = pd.DataFrame({
    "Method": ["Gemini 3 Flash", "SLM (Pass@1)", "SLM (Pass@5)"],
    "Accuracy (%)": [acc_gemini, acc_slm_pass1, acc_slm_pass5]
})

def add_bar_labels(ax, format_str='{:.2f}'):
    for p in ax.patches:
        ax.annotate(format_str.format(p.get_height()), 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', 
                    fontsize=11, color='black', xytext=(0, 5), 
                    textcoords='offset points')

plt.figure(figsize=(8, 6))
ax1 = sns.barplot(x="Method", y="Accuracy (%)", data=df_standard, palette=colors_3)
plt.title("Accuracy Comparison: Gemini vs. SLM", fontsize=14, pad=15)
plt.ylim(0, 105)
add_bar_labels(ax1)
plt.tight_layout()
plt.savefig("RO5_Accuracy_Comparison.png", dpi=300)
plt.show()

## 2. Accuracy by City Complexity

We load the test dataset to get the complexity (`num_cities`) for each task ID, and then group the accuracy results by complexity.

In [ ]:
# Mapping of task ID to complexity
with open('../data/trip_planning_test.json', 'r') as f:
    test_data = json.load(f)

id_to_complexity = {task_id: int(info['num_cities']) for task_id, info in test_data.items()}

data_list = []

for log in gemini_results.get('detailed_logs', []):
    task_id = log.get('id')
    complexity = id_to_complexity.get(task_id)
    is_correct = log.get('is_correct', False)
    data_list.append({
        "Method": "Gemini 3 Flash",
        "Complexity": complexity,
        "Accuracy": 100 if is_correct else 0
    })

for log in qwen_results.get('detailed_logs', []):
    task_id = log.get('id')
    complexity = id_to_complexity.get(task_id)
    pass1_correct = log.get('pass_1_success', False)
    pass5_correct = log.get('pass_5_success', False)
    data_list.append({
        "Method": "SLM (Pass@1)",
        "Complexity": complexity,
        "Accuracy": 100 if pass1_correct else 0
    })
    data_list.append({
        "Method": "SLM (Pass@5)",
        "Complexity": complexity,
        "Accuracy": 100 if pass5_correct else 0
    })

df_complexity = pd.DataFrame(data_list)
print(f"Total samples processed: {len(df_complexity)}")
df_complexity.head()

## 3. Visualization: Accuracy by City Complexity

In [ ]:
plt.figure(figsize=(12, 7))
ax2 = sns.barplot(
    x="Complexity", 
    y="Accuracy", 
    hue="Method", 
    data=df_complexity, 
    errorbar=None, 
    palette=colors_3
)

plt.title("Model Accuracy by Trip Complexity (Number of Cities)", fontsize=16, pad=20)
plt.xlabel("Complexity (Number of Cities)", fontsize=14)
plt.ylabel("Accuracy (%)", fontsize=14)
plt.ylim(0, 110)
plt.legend(title="Method", fontsize=12)

# Add data labels
for p in ax2.patches:
    if p.get_height() > 0:
        ax2.annotate(f'{p.get_height():.1f}%', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'bottom', 
                    xytext = (0, 5), 
                    textcoords = 'offset points', 
                    fontsize=10, 
                    fontweight='bold')

plt.tight_layout()
plt.savefig("accuracy_by_complexity.png", dpi=300)
plt.show()